# Clase 12 — Variables Categóricas, Encoding y Correlación

**Diplomado en Data Science Aplicada con Python para la Toma de Decisiones**  
Arca Continental Ecuador | UDLA

---

## Objetivos de hoy

1. **Variables categóricas**: identificar qué columnas NO son realmente numéricas
2. **Explorar categóricas**: frecuencias, proporciones, tablas cruzadas
3. **Encoding**: convertir categorías en números (Label, Ordinal, One-Hot)
4. **Correlación categórica**: Point-Biserial, Spearman, Cramér's V

**Dataset**: Titanic (seaborn).

## 0. Imports y carga de datos

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

df = sns.load_dataset("titanic")
print(f"{df.shape[0]} pasajeros, {df.shape[1]} columnas")
df.head()

In [ ]:
df.info()

---
## 1. ¿Todo número es un número?

Algunas columnas son `int64` o `float64` pero **no son numéricas de verdad**. 
Son categorías disfrazadas de números.

### La trampa: `df.describe()` las incluye como numéricas

In [ ]:
df.describe()

`pclass` y `survived` aparecen con media, std, etc. Pero...

In [ ]:
print("=== pclass (categorica disfrazada) ===")
print(f"Media: {df['pclass'].mean():.2f}")    # 2.31 -- clase 2.31 ???
print(f"Moda:  {df['pclass'].mode()[0]}")      # 3 (tercera clase)
print()
print("=== age (numerica real) ===")
print(f"Media:   {df['age'].mean():.2f}")      # 29.70 -- tiene sentido
print(f"Mediana: {df['age'].median():.2f}")     # 28.00 -- tiene sentido

**Conclusión**: la media de `pclass` = 2.31 no significa nada. ¿"Clase 2.31"? 
Lo mismo con `survived`: ¿"desviación estándar de estar vivo"?

**Que una variable esté guardada como número NO significa que sea numérica.**

### Tipos de variables

| Tipo | Definición | Ejemplo Titanic | Operaciones válidas |
|------|-----------|-----------------|--------------------|
| **Nominal** | Categorías sin orden | sex, embarked | moda, frecuencia |
| **Ordinal** | Categorías con orden | class (1ra > 2da > 3ra) | moda, frecuencia, mediana |
| **Binaria** | Solo dos valores | survived, alone | moda, proporción |
| **Discreta** | Números contables | sibsp, parch | media, mediana, std |
| **Continua** | Números medibles | age, fare | media, mediana, std |

### Clasificar cada columna del Titanic

In [ ]:
for col in df.columns:
    print(f"{col:15s}  dtype={str(df[col].dtype):10s}  "
          f"nunique={df[col].nunique():3d}  "
          f"ejemplo: {df[col].dropna().iloc[0]}")

| Columna | dtype | Tipo real | ¿Por qué? |
|---------|-------|-----------|----------|
| survived | int64 | Binaria | 0/1 = murió/sobrevivió |
| pclass | int64 | **Ordinal** | 1, 2, 3 = clase social (¡no es numérica!) |
| sex | object | Nominal | male/female, sin orden |
| age | float64 | Continua | Años de vida |
| sibsp | int64 | Discreta | Número de hermanos/esposos |
| parch | int64 | Discreta | Número de padres/hijos |
| fare | float64 | Continua | Precio del boleto |
| embarked | object | Nominal | C/Q/S = puertos |
| class | category | Ordinal | First/Second/Third |
| who | object | Nominal | man/woman/child |
| deck | category | Ordinal | A-G (cubiertas del barco) |
| alive | object | Binaria | yes/no |

---
## 2. Explorar variables categóricas correctamente

Para categóricas NO usamos media ni std. Usamos **frecuencias y proporciones**.

### `value_counts()` — frecuencias absolutas y relativas

In [ ]:
print("--- Frecuencias absolutas ---")
print(df["embarked"].value_counts())
print()
print("--- Proporciones ---")
print(df["embarked"].value_counts(normalize=True).round(3))

In [ ]:
print(df["sex"].value_counts())
print()
print(df["class"].value_counts())

### `countplot` — visualizar frecuencias

In [ ]:
sns.countplot(x="embarked", data=df)
plt.title("Pasajeros por Puerto de Embarque")
plt.show()

In [ ]:
sns.countplot(x="class", data=df)
plt.title("Pasajeros por Clase")
plt.show()

### `pd.crosstab()` — cruzar dos categóricas

Ya lo vimos en la clase 11. Ahora lo usamos para responder:
**¿La supervivencia depende del sexo?**

In [ ]:
pd.crosstab(df["sex"], df["survived"], margins=True)

In [ ]:
pd.crosstab(df["sex"], df["survived"], normalize="index").round(3)

**74.2% de mujeres sobrevivieron vs 18.9% de hombres.** 
"Women and children first" se confirma en los datos.

In [ ]:
pd.crosstab(df["class"], df["survived"], normalize="index").round(3)

La primera clase tiene la mayor tasa de supervivencia (63%), la tercera la menor (24%).

---
## 3. Encoding: convertir categorías en números

Las computadoras necesitan números para calcular correlación, hacer modelos, etc. 
Pero **no todos los encodings sirven para todo**.

### 3.1 Label Encoding — para variables binarias

Asignar 0 y 1 a las dos categorías. Solo funciona cuando hay exactamente dos valores.

In [ ]:
df["sex_encoded"] = df["sex"].map({"male": 0, "female": 1})

df[["sex", "sex_encoded"]].head()

In [ ]:
df["alive_encoded"] = df["alive"].map({"no": 0, "yes": 1})

df[["alive", "alive_encoded"]].head()

### 3.2 Ordinal Encoding — para variables con orden

Asignar números que respeten el orden natural de las categorías.

In [ ]:
df["class_encoded"] = df["class"].map({
    "Third": 1,
    "Second": 2,
    "First": 3
})

df[["class", "class_encoded"]].head()

Los números respetan el orden: Third (1) < Second (2) < First (3).

**Nota**: `pclass` ya viene como número (1, 2, 3) pero al revés: 1 = First. 
Es una ordinal que ya está codificada.

### 3.3 One-Hot Encoding — para variables nominales

Cuando NO hay orden entre las categorías, crear una columna por cada categoría.

**¿Por qué no usar Label?** Si codificamos embarked como C=0, Q=1, S=2, 
el algoritmo "cree" que S > Q > C y que las distancias son iguales. 
¡Ambas cosas son falsas!

In [ ]:
dummies = pd.get_dummies(df["embarked"], prefix="emb")
dummies.head()

Cada categoría se convierte en su propia columna (True/False = 1/0).

In [ ]:
dummies_drop = pd.get_dummies(df["embarked"], prefix="emb", drop_first=True)
dummies_drop.head()

`drop_first=True` elimina la primera columna (C). Si `emb_Q=0` y `emb_S=0`, sabemos que es C. 
Esto evita redundancia.

In [ ]:
who_dummies = pd.get_dummies(df["who"], prefix="who", drop_first=True)
who_dummies.head()

### Resumen: ¿Cuándo usar cada encoding?

| Tipo de variable | Encoding | Ejemplo |
|-----------------|----------|--------|
| **Binaria** (2 valores) | Label (0/1) | sex → 0/1 |
| **Ordinal** (con orden) | Ordinal (números con orden) | class → 1, 2, 3 |
| **Nominal** (sin orden) | One-Hot (dummies) | embarked → emb_C, emb_Q, emb_S |

**Si tienes duda entre ordinal y nominal, usa One-Hot: es más seguro.**

---
## 4. Correlación con variables categóricas

Ya sabemos Pearson y Spearman de la clase 11. Pero...
**¿Puedo hacer `.corr()` después del encoding?**

**Depende del tipo de variable:**

| Combinación | ¿Pearson? | ¿Qué usar? |
|------------|-----------|------------|
| Binaria (0/1) vs Numérica | Sí | Point-Biserial (= Pearson con 0/1) |
| Binaria vs Binaria | Sí | Phi (= Pearson con dos binarias) |
| Ordinal vs Numérica | No ideal | Spearman |
| Nominal vs cualquiera | **No** | Cramér's V |

### 4.1 Point-Biserial: Pearson con una binaria

Cuando una variable es binaria (0/1) y la otra numérica, 
Pearson funciona y se llama **Point-Biserial**. 
Es la misma fórmula, el mismo `.corr()`. No hay nada nuevo.

In [ ]:
cols_binarias = ["survived", "sex_encoded", "age", "fare"]
df[cols_binarias].corr().round(3)

- `sex_encoded` vs `survived` = 0.54: **ser mujer se asocia fuertemente con sobrevivir.**
- `fare` vs `survived` = 0.26: pagar más se asocia moderadamente con sobrevivir.
- `age` vs `survived` = -0.08: la edad casi no tiene correlación con sobrevivir.

In [ ]:
sns.heatmap(df[cols_binarias].corr(), annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, vmin=-1, vmax=1)
plt.title("Correlacion (Point-Biserial) - Variables Binarias")
plt.tight_layout()
plt.show()

### 4.2 Spearman para variables ordinales

Para ordinales, **Spearman** es mejor porque solo usa el **orden** (rangos), 
no asume que las distancias entre categorías son iguales.

In [ ]:
cols_ordinal = ["survived", "class_encoded", "fare", "age"]
df[cols_ordinal].corr(method="spearman").round(3)

- `class_encoded` vs `survived` = 0.34: clase más alta → más supervivencia.
- `class_encoded` vs `fare` = 0.89: primera clase cuesta más (lógico).

### 4.3 Cramér's V: dos categóricas nominales

Cuando ambas variables son categóricas nominales, **ni Pearson ni Spearman funcionan**.

Cramér's V mide la asociación entre dos categóricas:
- Va de **0** (sin asociación) a **1** (asociación perfecta)
- No tiene dirección (no es positivo ni negativo)

**¿Cómo funciona?**

1. Construye la tabla de contingencia (crosstab) con frecuencias **observadas** ($O_{ij}$)
2. Calcula las frecuencias **esperadas** ($E_{ij}$) si no hubiera relación:

$$E_{ij} = \frac{\text{total fila}_i \times \text{total columna}_j}{\text{total general}}$$

3. Calcula **chi-cuadrado** ($\chi^2$): cuánto difieren observadas vs esperadas:

$$\chi^2 = \sum_{i,j} \frac{(O_{ij} - E_{ij})^2}{E_{ij}}$$

4. Normaliza para obtener un valor entre 0 y 1:

$$V = \sqrt{\frac{\chi^2}{n \cdot (k - 1)}}$$

donde $n$ = total de observaciones y $k$ = min(filas, columnas).

In [ ]:
def cramers_v(col1, col2):
    """Calcula Cramer's V entre dos variables categoricas."""
    tabla = pd.crosstab(col1, col2)
    chi2 = chi2_contingency(tabla)[0]
    n = len(col1)
    k = min(tabla.shape) - 1
    return (chi2 / (n * k)) ** 0.5

In [ ]:
v = cramers_v(df["sex"], df["survived"].map({0: "No", 1: "Si"}))
print(f"Cramer's V (sex vs survived): {v:.3f}")

V ≈ 0.54 → asociación **fuerte** entre sexo y supervivencia.

Interpretación general:
- 0 – 0.1: débil
- 0.1 – 0.3: moderada
- \> 0.3: fuerte

In [ ]:
v_emb = cramers_v(df["embarked"].dropna(),
                  df.loc[df["embarked"].notna(), "survived"].map({0: "No", 1: "Si"}))
print(f"Cramer's V (embarked vs survived): {v_emb:.3f}")

In [ ]:
v_class = cramers_v(df["class"], df["survived"].map({0: "No", 1: "Si"}))
print(f"Cramer's V (class vs survived): {v_class:.3f}")

### 4.4 Heatmap de correlación con variables codificadas

Ahora que tenemos las variables codificadas, podemos hacer un heatmap 
que incluya **tanto numéricas como categóricas** (ya convertidas).

Usamos Spearman porque incluye ordinales.

In [ ]:
cols_heatmap = ["survived", "sex_encoded", "class_encoded",
                "age", "fare", "sibsp", "parch"]

plt.figure(figsize=(8, 6))
sns.heatmap(df[cols_heatmap].corr(method="spearman"),
            annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, vmin=-1, vmax=1)
plt.title("Correlacion (Spearman) - Titanic")
plt.tight_layout()
plt.show()

### ¿Qué predice la supervivencia?

| Variable | Tipo | Correlación con survived | Método |
|---------|------|-------------------------|--------|
| sex (ser mujer) | Binaria | ≈ +0.54 | Point-Biserial |
| class (1ra clase) | Ordinal | ≈ +0.34 | Spearman |
| fare (tarifa alta) | Continua | ≈ +0.33 | Spearman |
| embarked | Nominal | V ≈ 0.18 | Cramér's V |
| age | Continua | ≈ −0.06 | Spearman |

**Ser mujer y viajar en primera clase** son los factores más asociados 
con la supervivencia.

---
## 5. Ejercicios

### Ejercicio 1: Identificar tipos de variables

¿`sibsp` (número de hermanos/esposos a bordo) es categórica o numérica? 
¿Tiene sentido calcular su media? ¿Y su moda?

In [ ]:
# Tu respuesta aqui
print(f"Media de sibsp: {df['sibsp'].mean():.2f}")
print(f"Moda de sibsp:  {df['sibsp'].mode()[0]}")
print()
print(df["sibsp"].value_counts())

### Ejercicio 2: Encoding correcto

Aplica el encoding correcto a `who` (man/woman/child). 
¿Es Label, Ordinal o One-Hot? ¿Por qué?

In [ ]:
# Tip: who tiene 3 categorias: man, woman, child
# Hay un orden natural entre ellas?

# Tu codigo aqui:


### Ejercicio 3: Crosstab y Cramér's V

1. Haz un `crosstab` de `embarked` vs `survived` con proporciones por fila.
2. ¿Cuál puerto tiene la mayor tasa de supervivencia?
3. Calcula el Cramér's V entre `who` y `survived`. ¿Es fuerte?

In [ ]:
# 1. Crosstab embarked vs survived


In [ ]:
# 2. Cramer's V entre who y survived


### Ejercicio 4: ¿Tiene sentido esta operación?

Para cada caso, responde si tiene sentido o no y por qué:

1. `df['pclass'].mean()`
2. `df['fare'].median()`
3. `df['embarked'].mode()`
4. `df['survived'].std()`
5. `df['age'].value_counts()`

In [ ]:
# Escribe tus respuestas aqui:
# 1. 
# 2. 
# 3. 
# 4. 
# 5. 
